# 🧪 AiStock 组合管理测试

## 测试目标
- ✅ PortfolioTracker: 持仓跟踪
- ✅ Rebalancer: 再平衡策略
- ✅ RiskManager: 风险管理

## 测试内容
- 买卖操作
- 盈亏计算
- 权重偏离检测
- 止损止盈预警

In [ ]:
# 环境设置
import sys
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print("✅ 环境设置完成")

In [ ]:
# 导入组合管理模块
from base_services.config_service import ConfigService
from dynamic_price_system.portfolio.tracker import PortfolioTracker
from dynamic_price_system.portfolio.rebalancer import Rebalancer
from dynamic_price_system.portfolio.risk_manager import RiskManager

print("✅ 组合管理模块导入成功")

## 1️⃣ 持仓跟踪测试

In [ ]:
# 初始化持仓跟踪器
config = ConfigService(system_name='dynamic_price')
tracker = PortfolioTracker(initial_capital=1000000, config=config)

print("✅ 持仓跟踪器初始化成功")
print(f"📊 初始资金：¥{tracker.initial_capital:,.0f}")

In [ ]:
# 测试买入操作
print("\n📊 测试买入操作：")

trades = [
    ('600938', 40.00, 5000),   # 中国海油
    ('601899', 32.00, 8000),   # 紫金矿业
    ('300750', 400.00, 500),   # 宁德时代
    ('600406', 30.00, 3000),   # 国电南瑞
]

for code, price, qty in trades:
    success = tracker.buy(code, price, qty)
    status = "✅" if success else "❌"
    print(f"  {status} 买入 {code} {qty}股 @ ¥{price:.2f}")

print(f"\n📊 剩余现金：¥{tracker.cash:,.2f}")

In [ ]:
# 测试组合摘要
current_prices = {
    '600938': 42.24,
    '601899': 32.40,
    '300750': 400.50,
    '600406': 30.70
}

summary = tracker.get_summary(current_prices)

print("\n📊 组合摘要：")
print(f"  • 总市值：¥{summary['total_value']:,.2f}")
print(f"  • 现金：¥{summary['cash']:,.2f}")
print(f"  • 总收益：{summary['total_return']}")
print(f"  • 交易次数：{summary['transaction_count']}")

print("\n📊 持仓明细：")
for pos in summary['positions']:
    print(f"  • {pos['code']}: {pos['quantity']}股 | 盈利{pos['profit_pct']} | 权重{pos['weight']}")

## 2️⃣ 再平衡测试

In [ ]:
# 测试再平衡策略
rebalancer = Rebalancer(config)

# 目标权重
target_weights = {
    '600938': 0.15,
    '601899': 0.10,
    '300750': 0.12,
    '600406': 0.12
}

total_value = tracker.get_portfolio_value(current_prices)
current_positions = tracker.positions

actions = rebalancer.check_rebalance(
    current_positions=current_positions,
    current_prices=current_prices,
    target_weights=target_weights,
    total_value=total_value
)

print("\n📊 再平衡建议：")
if actions:
    for action in actions:
        print(f"  • {action['code']} {action['action']} {action['quantity']}股")
        print(f"    原因：{action['reason']}")
        print(f"    当前权重：{action['current_weight']} → 目标：{action['target_weight']}")
else:
    print("  ✅ 无需再平衡")

In [ ]:
# 测试资金约束优化
optimized = rebalancer.optimize_actions(actions, available_cash=tracker.cash)

print(f"\n📊 优化后执行 {len(optimized)}/{len(actions)} 个动作")
for action in optimized:
    note = action.get('note', '')
    if note:
        print(f"  ⚠️ {action['code']}: {note}")

## 3️⃣ 风险管理测试

In [ ]:
# 测试风险管理
risk_manager = RiskManager(config, portfolio=tracker)

# 模拟动态价格结果
dynamic_prices = [
    {
        'code': '600938',
        'sector': '油气开采',
        'current_price': 42.24,
        'stop_loss': 39.20,
        'target_price': 48.50
    },
    {
        'code': '601899',
        'sector': '黄金',
        'current_price': 32.40,
        'stop_loss': 30.15,
        'target_price': 41.00
    },
    {
        'code': '300750',
        'sector': '新能源',
        'current_price': 400.50,
        'stop_loss': 360.45,
        'target_price': 480.00
    },
    {
        'code': '600406',
        'sector': '特高压',
        'current_price': 30.70,
        'stop_loss': 28.25,
        'target_price': 36.00
    }
]

alerts = risk_manager.check_alerts(
    dynamic_prices=dynamic_prices,
    current_prices=current_prices,
    positions=tracker.positions
)

print("\n📊 风险预警：")
if alerts:
    for alert in alerts:
        print(f"  [{alert['level']}] {alert['message']}")
else:
    print("  ✅ 无风险预警")

In [ ]:
# 生成风险报告
report = risk_manager.generate_risk_report(alerts, summary)

print("\n📊 风险报告：")
print(f"  • 总预警数：{report['total_alerts']}")
print(f"  • 高危：{report['high_priority']}")
print(f"  • 中危：{report['medium_priority']}")
print(f"  • 风险等级：{report['risk_level']}")

## 4️⃣ 盈亏分析

In [ ]:
# 测试不同价格场景下的盈亏
scenarios = [
    {'name': '当前价格', 'prices': current_prices},
    {'name': '上涨 10%', 'prices': {k: v*1.1 for k, v in current_prices.items()}},
    {'name': '下跌 10%', 'prices': {k: v*0.9 for k, v in current_prices.items()}},
    {'name': '达到目标价', 'prices': {r['code']: r['target_price'] for r in dynamic_prices}},
    {'name': '触及止损价', 'prices': {r['code']: r['stop_loss'] for r in dynamic_prices}},
]

print("\n📊 盈亏情景分析：")
for scenario in scenarios:
    return_rate = tracker.get_portfolio_return(scenario['prices'])
    total_value = tracker.get_portfolio_value(scenario['prices'])
    print(f"  • {scenario['name']}: 收益率{return_rate:.1%} | 总值¥{total_value:,.0f}")

## 📊 测试总结

In [ ]:
print("="*60)
print("📋 组合管理测试总结")
print("="*60)
print(f"✅ PortfolioTracker: 持仓跟踪正常")
print(f"✅ 交易次数：{summary['transaction_count']}")
print(f"✅ 当前收益：{summary['total_return']}")
print(f"✅ Rebalancer: 再平衡策略正常")
print(f"✅ RiskManager: 风险管理正常")
print(f"✅ 风险等级：{report['risk_level']}")
print("="*60)